# UNet 训练（Kaggle T4 版）— cascade 方案

**与本地 `train_unet.py` 共用同一份脚本**：notebook 通过 subprocess 调用脚本，避免逻辑二份化。

## 准备工作
1. 在 Kaggle 上传两个 dataset：
   - `rectal-cancer-data` ← 本仓库 `直肠癌数据/` 全量 DICOM
   - `ctai-code-and-splits` ← 本仓库 `ctai_model_code/`（**不要包含 yolo_dataset/** 太大，splits.json 必须在内）
2. Notebook 选 **GPU T4 ×2** 加速器，开 internet（首次需要装 SimpleITK / albumentations / pydicom）
3. 默认 `SMOKE_TEST = True`：先跑 5 epoch 烟雾测试。验证通过后改成 `False`，重启 notebook 跑完整 200 epoch。

## 输出
- `/kaggle/working/checkpoints/unet_best_ema.pth`（含完整 metadata，single weights source，约 126 MB）
- `/kaggle/working/checkpoints/training_curves.png`
- `/kaggle/working/checkpoints/training_log.csv`
- `/kaggle/working/train_log_full.txt`（subprocess 完整 stdout，调试用；最后 100 行也会打到本 cell）
- `/kaggle/working/unet_output.zip`（一键打包供下载，含 train_log_full.txt）

In [ ]:
# === Cell 1: 环境 + 依赖安装 ===
import os, sys, glob, subprocess, shutil
print('python  :', sys.version.split()[0])
print('cwd     :', os.getcwd())
# Kaggle 已预装所需依赖（SimpleITK / albumentations / pydicom），跳过 pip install 避免 DNS 失败
print('[INFO] Skipped pip install — using Kaggle pre-installed packages')
import torch
print('torch   :', torch.__version__, '| CUDA:', torch.cuda.is_available(),
      '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# === Cell 2: 定位 datasets，校验 splits fingerprint ===
import json
EXPECTED_FP = '84706e8b75c8f403'
KAGGLE_NS = '/kaggle/input/datasets/ramyaramyarao'

# 数据目录
DATA_CANDIDATES = [
    f'{KAGGLE_NS}/rectal-cancer-data/直肠癌数据',
    '/kaggle/input/rectal-cancer-data/直肠癌数据',
    '/kaggle/input/rectal-cancer-data/rectal_data',
    '/kaggle/input/rectal-cancer-data',
]
DATA_DIR = next((p for p in DATA_CANDIDATES if os.path.isdir(p)), None)
if DATA_DIR is None:
    raise SystemExit('找不到数据目录，请检查 rectal-cancer-data dataset 是否上传')
n_patients = len([d for d in os.listdir(DATA_DIR) if d.isdigit()])
print(f'DATA_DIR = {DATA_DIR}  (患者数 {n_patients})')

# 代码目录（含 train_unet.py 与 splits.json）
CODE_DIR_CANDIDATES = [f'{KAGGLE_NS}/ctai-code-and-splits/ctai_model_code'] + \
    glob.glob('/kaggle/input/**/ctai_model_code/train_unet.py', recursive=True)
CODE_DIR = None
for c in CODE_DIR_CANDIDATES:
    if os.path.isdir(c) and os.path.exists(os.path.join(c, 'train_unet.py')):
        CODE_DIR = c
        break
    elif c.endswith('train_unet.py'):
        CODE_DIR = os.path.dirname(c)
        break
if CODE_DIR is None:
    raise SystemExit('找不到 train_unet.py，请上传 ctai-code-and-splits dataset')
SPLITS_JSON = os.path.join(CODE_DIR, 'splits.json')
TRAIN_SCRIPT = os.path.join(CODE_DIR, 'train_unet.py')
print(f'CODE_DIR     = {CODE_DIR}')
print(f'TRAIN_SCRIPT = {TRAIN_SCRIPT}')
print(f'SPLITS_JSON  = {SPLITS_JSON}')

# 校验 fingerprint
with open(SPLITS_JSON, 'r', encoding='utf-8') as f:
    payload = json.load(f)
assert payload['fingerprint'] == EXPECTED_FP, \
    f"split fingerprint mismatch: {payload['fingerprint']!r} != {EXPECTED_FP!r}"
print(f"splits OK: fp={payload['fingerprint']}, "
      f"train={len(payload['splits']['train'])} | val={len(payload['splits']['val'])} | "
      f"test={len(payload['splits']['test'])}")

In [ ]:
# === Cell 3: 训练模式开关 ===
# False = 正式 200 epoch 训练（~6-10 h，T4×2）
# True  = 5 epoch smoke 测试（~15 min）
SMOKE_TEST = False

WORK = '/kaggle/working'
SAVE_DIR = os.path.join(WORK, 'checkpoints')
LOG_FILE = os.path.join(WORK, 'train_log_full.txt')
os.makedirs(SAVE_DIR, exist_ok=True)

print(f'SMOKE_TEST = {SMOKE_TEST}')
print(f'SAVE_DIR   = {SAVE_DIR}')
print(f'LOG_FILE   = {LOG_FILE}')

In [ ]:
# === Cell 4: 启动训练（stdout 重定向到文件 + 每 1s 回显进度到 Notebook）===
import time, threading

cmd = [
    sys.executable, '-u', TRAIN_SCRIPT,         # -u: unbuffered stdout，及时 flush 到文件
    '--splits_json', SPLITS_JSON,
    '--data_dir',    DATA_DIR,
    '--save_dir',    SAVE_DIR,
    '--num_workers', '2',
]
if SMOKE_TEST:
    cmd += ['--smoke_test', '--epochs', '5']
    print('[mode] SMOKE_TEST — 5 epoch 烟雾测试')
else:
    print('[mode] FULL TRAINING — 200 epoch')
print('[cmd ]', ' '.join(cmd))
print(f'[log ] 完整 stdout 写入 {LOG_FILE}')
print(f'[mon ] 每 1s 打印最新进度 ↓')
print('---')

t0 = time.time()
with open(LOG_FILE, 'w', encoding='utf-8') as logf:
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True, bufsize=1)

    # 后台线程：每 1s 从日志尾部取一行打到 notebook
    stop_monitor = threading.Event()
    last_tail = ['']

    def monitor():
        while not stop_monitor.is_set():
            stop_monitor.wait(1)
            try:
                with open(LOG_FILE, 'r', encoding='utf-8', errors='replace') as lf:
                    lines = [l.strip() for l in lf.readlines() if l.strip()]
                if lines:
                    latest = lines[-1]
                    if latest != last_tail[0]:
                        print(f'  [{time.strftime("%H:%M:%S")}] {latest[:200]}', flush=True)
                        last_tail[0] = latest
            except Exception:
                pass

    mon_thread = threading.Thread(target=monitor, daemon=True)
    mon_thread.start()

    # 主线程：逐行写入日志文件
    for line in proc.stdout:
        logf.write(line)
        logf.flush()

    ret = proc.wait()
    stop_monitor.set()
    mon_thread.join(timeout=5)

dur = time.time() - t0
print(f'\n[done] training subprocess exit={ret}  wall={dur/60:.1f} min')

# 读最后 N 行打印到 Notebook
TAIL_N = 100
with open(LOG_FILE, 'r', encoding='utf-8', errors='replace') as f:
    lines = f.readlines()
print(f'\n=== train_log_full.txt 总行数 {len(lines)}，下面是最后 {min(TAIL_N, len(lines))} 行 ===\n')
print(''.join(lines[-TAIL_N:]))
if ret != 0:
    raise SystemExit(f'training exited with code {ret}（详见 {LOG_FILE}）')

In [ ]:
# === Cell 5: 校验产物 + 打包供下载 ===
expected = ['unet_best_ema.pth', 'best_model.pth', 'training_log.csv']
missing = [f for f in expected if not os.path.exists(os.path.join(SAVE_DIR, f))]
if missing:
    print(f'[WARN] 缺少产物: {missing}')

# 校验 metadata + size sanity
best_path = os.path.join(SAVE_DIR, 'unet_best_ema.pth')
if os.path.exists(best_path):
    size_mb = os.path.getsize(best_path) / 1e6
    ckpt = torch.load(best_path, map_location='cpu', weights_only=False)
    print('=== unet_best_ema.pth metadata ===')
    for k in ['best_epoch', 'last_epoch', 'best_dice', 'weights_kind',
              'split_fingerprint', 'data_dir', 'timestamp', 'git_commit',
              'produced_by']:
        print(f'  {k:20s} = {ckpt.get(k)!r}')
    print(f'  patient_ids_train (n)= {len(ckpt.get("patient_ids_train", []))}')
    print(f'  patient_ids_val   (n)= {len(ckpt.get("patient_ids_val", []))}')
    print(f'  size_MB           = {size_mb:.1f}')

    # ---- 硬性校验 ----
    assert ckpt['split_fingerprint'] == EXPECTED_FP, 'fingerprint 写入异常！'
    assert 'best_epoch' in ckpt and 'last_epoch' in ckpt, \
        'metadata 缺 best_epoch / last_epoch — 检查 train_unet.py 是否使用了旧版 repackage'
    assert ckpt.get('weights_kind') in ('ema', 'raw'), \
        f"weights_kind 异常: {ckpt.get('weights_kind')!r}"
    # 体积 sanity：单一权重源 ≈ 126 MB；放宽到 100~180 MB 以覆盖不同 attention_unet 变体
    assert size_mb < 180, f'unet_best_ema.pth 体积 {size_mb:.1f} MB 异常（>180）—— 可能 optimizer / 重复权重未剥离'
    if ckpt['weights_kind'] != 'ema':
        print("[WARN] weights_kind != 'ema'，best_model.pth 中无 EMA 副本，回退到 raw model_state_dict")
    if (ckpt['best_epoch'] in (0, 1, None)) and (ckpt['last_epoch'] or 0) >= 5:
        print(f"[WARN] best_epoch={ckpt['best_epoch']} last_epoch={ckpt['last_epoch']} "
              f"—— 训练 {ckpt['last_epoch']} epoch 但 best 卡在起点；smoke 模式正常，full 模式必须排查")
    print('[OK] metadata + size 校验通过')

# 把 train_log_full.txt 一起打进包（后续排查用）
if os.path.exists(LOG_FILE):
    shutil.copy2(LOG_FILE, os.path.join(SAVE_DIR, 'train_log_full.txt'))

shutil.make_archive(os.path.join(WORK, 'unet_output'), 'zip', SAVE_DIR)
print('=== 输出文件清单 ===')
for f in sorted(os.listdir(SAVE_DIR)):
    p = os.path.join(SAVE_DIR, f)
    if os.path.isfile(p):
        print(f'  {f}  ({os.path.getsize(p)/1e6:.1f} MB)')
print(f'=> /kaggle/working/unet_output.zip 已生成，到 Notebook 输出区下载')